In [1]:
pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [8]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# ==== 1) Veriyi yükle ====
CSV_PATH = "Epey_laptop_clean_elle.csv"  # dosya adı bu çalışmaya göre ayarlandı
df = pd.read_csv(CSV_PATH)

# ==== 2) Hedefi ve kullanılmayacak kolonları belirle ====
TARGET = "price_text"

# Modelde kullanılmaması gereken kolonlar
drop_cols = [c for c in ["id", "url", "title", "Unnamed: 0"] if c in df.columns]

# Hedefi boş olan satırları at
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

# ==== 3) Özellik/etiket ayrımı ====
X = df.drop(columns=[TARGET] + drop_cols, errors="ignore")
y = df[TARGET].astype(float)

# 3.a) Kategorik kolonlar (object/string/category)
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

# 3.b) CatBoost kategoriklerde NaN görmek istemez -> string yapıp 'NA' doldur
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

# (İsteğe bağlı) sayısal görünümlü stringleri sayıya çevir
for c in cat_cols:
    s = X[c].str.replace(",", ".", regex=False)
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.9:
        X[c] = num

# cat_cols’u güncelle (string kalanlar kategorik)
cat_cols = X.select_dtypes(include=["string", "object", "category"]).columns.tolist()

print(f"Kategorik kolonlar ({len(cat_cols)}): {cat_cols}")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool = Pool(X_test,  y_test,  cat_features=cat_cols)

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=3000,
    random_seed=42,
    l2_leaf_reg=3.0,
    od_type="Iter",
    od_wait=200,
    verbose=200
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

preds = model.predict(valid_pool)
mae  = mean_absolute_error(y_test, preds)
mse  = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
y_true = np.asarray(y_test, dtype=float)
mask = y_true != 0
mape = np.mean(np.abs((y_true[mask] - preds[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
r2   = r2_score(y_test, preds)

print(f"MAE : {mae:,.2f} TL")
print(f"RMSE: {rmse:,.2f} TL")
print(f"MAPE: {mape:,.2f} %")
print(f"R²  : {r2:,.4f}")

# ==== 7) Özellik önemleri ====
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance(train_pool)
}).sort_values("importance", ascending=False)

print("\nÖzellik önemleri (ilk 20):")
print(fi.head(20).to_string(index=False))

# CSV olarak kaydet
#fi.to_csv("feature_importances.csv", index=False, encoding="utf-8-sig")

# ==== 8) Modeli kaydet ====
os.makedirs("models", exist_ok=True)
model_path_cbm = "models/catboost_price.cbm"
model_path_pkl = "models/catboost_price.pkl"
model.save_model(model_path_cbm)
joblib.dump(model, model_path_pkl)
print(f"\nModel kaydedildi: {model_path_cbm} ve {model_path_pkl}")

# ==== 9) Örnek tahmin ====
sample = X_test.iloc[:5].copy()
sample_pred = model.predict(sample)
print("\nÖrnek tahminler (ilk 5):")
print(pd.DataFrame({"y_true": y_test.iloc[:5].values, "y_pred": sample_pred}))


Kategorik kolonlar (6): ['Ürün Amacı', 'Ürün Ailesi', 'İşlemci Modeli', 'Bellek Türü', 'GPU Modeli', 'GPU Bellek Türü']
0:	learn: 65666.6540867	test: 74548.1268047	best: 74548.1268047 (0)	total: 24.1ms	remaining: 1m 12s
200:	learn: 6188.0645318	test: 31176.8072243	best: 31176.8072243 (200)	total: 7.14s	remaining: 1m 39s
400:	learn: 966.0672610	test: 29268.8537754	best: 29268.8537754 (400)	total: 14.6s	remaining: 1m 34s
600:	learn: 275.0252729	test: 29207.6388794	best: 29207.6388794 (600)	total: 22.1s	remaining: 1m 28s
800:	learn: 86.8991870	test: 29211.3802826	best: 29206.5166350 (609)	total: 30.1s	remaining: 1m 22s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 29206.51663
bestIteration = 609

Shrink model to first 610 iterations.
MAE : 20,741.20 TL
RMSE: 29,206.52 TL
MAPE: 12.42 %
R²  : 0.8452

Özellik önemleri (ilk 20):
                feature  importance
Sabit Disk (SSD) Boyutu   15.122951
            Marka Puanı   14.576181
           Ekran Boyutu   13.633552
 

In [ ]:
#GPU Model,Ürün ailesi(Marka puanı),İşlemci ,SSD,Ekran boyutu ,RAM ,Ekran Yenileme hızı,Ürün amacı,Bellek türü

In [ ]:
from catboost import CatBoostRegressor, Pool

train_pool = Pool(X, y, cat_features=cat_cols)

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=3000,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=200
)

model.fit(train_pool)  



In [ ]:
import pandas as pd
from catboost import Pool
import joblib

model = joblib.load("catboost_price_full.pkl")
feature_cols = joblib.load("feature_columns.pkl")
cat_cols_known = joblib.load("cat_cols.pkl")

df_new = pd.DataFrame([tahmin.csv])  

X_new = df_new.copy()
for c in set(cat_cols_known).intersection(X_new.columns):
    X_new[c] = X_new[c].astype("string").fillna("NA")

for col in feature_cols:
    if col not in X_new.columns:
        X_new[col] = "NA" if col in cat_cols_known else 0
X_new = X_new[feature_cols]

cat_idx = [i for i, c in enumerate(feature_cols) if c in cat_cols_known]
pool_new = Pool(X_new, cat_features=cat_idx)

pred = model.predict(pool_new)
print(pred)
